# 07 • Dados reais de comércio: diagnóstico e regras

## Preparar esta sessão
Execute as células abaixo antes de começar a aula. Elas localizam os arquivos e preparam os objetos necessários. Você não precisa decorar essa preparação agora.

In [ ]:
from pathlib import Path
import pandas as pd
import math
BASE = Path.cwd()
if not (BASE / 'dados').exists() and (BASE.parent / 'dados').exists():
    BASE = BASE.parent
DADOS = BASE / 'dados'
SAIDAS = BASE / 'resultados'
SAIDAS.mkdir(exist_ok=True)
assert (DADOS/'penguins.csv').exists(), 'Descompacte o curso inteiro e abra na pasta correta.'


## Preparação independente
Execute esta célula para recriar os objetos necessários dos módulos anteriores.

In [ ]:
p = pd.read_csv(DADOS / 'penguins.csv')
print(p.head(3).to_string(index=False))
print('Dimensões:', p.shape)
print('Colunas:', p.columns.tolist())
print(type(p['species']).__name__)

mascara = p['species'].eq('Adelie') & p['body_mass_g'].gt(4000)
print(p.loc[mascara, ['species', 'island', 'body_mass_g']].head())
print(p.iloc[:3, :2])

qualidade = pd.DataFrame({'ausentes':p.isna().sum(), 'percentual':p.isna().mean().mul(100)})
print(qualidade)
medidos = p.dropna(subset=['body_mass_g']).copy()
print('Observações com massa:', len(medidos))

enriquecido = p.assign(body_mass_kg=p['body_mass_g']/1000)
print(enriquecido[['body_mass_g','body_mass_kg']].head())
print(p['species'].astype('string').str.lower().head())

resumo = p.groupby('species', as_index=False, observed=True).agg(
    observacoes=('body_mass_g','size'),
    massas_validas=('body_mass_g','count'),
    massa_media_g=('body_mass_g','mean'))
print(resumo)

nomes = pd.DataFrame({'species':['Adelie','Chinstrap','Gentoo'],
                      'nome_pt':['Adélia','Barbicha','Papua']})
cruzado = p.merge(nomes, on='species', how='left', validate='many_to_one', indicator=True)
print(cruzado['_merge'].value_counts())

### Leitura comentada da regra comercial

Comece pela inspeção: `r.head()`, `r.info()` e `r.isna().sum()`. Não calcule receita antes de saber o que cada linha significa. Uma fatura com três produtos pode aparecer em três linhas; portanto len(r) não conta pedidos. `r['InvoiceNo'].nunique()` conta códigos distintos de fatura.

No exemplo, cancelada é uma máscara. A sequência str.upper().str.startswith('C', na=False) padroniza o código para maiúsculas e pergunta se começa com C. elegivel combina quatro exigências com &: quantidade positiva, preço positivo, data presente e ausência de cancelamento. O símbolo ~ inverte cancelada. Em seguida, loc mantém somente as linhas aprovadas, copy cria a tabela de trabalho e a multiplicação calcula o valor de cada item.

Exemplo fictício para entender a conta: uma fatura com duas linhas, uma de 2 unidades a £3 e outra de 1 unidade a £4, vale £10. A média dos valores das linhas é £5, mas o ticket dessa fatura é £10. Por isso o ticket médio exige primeiro somar itens por fatura e depois calcular a média dos totais de fatura. Não transfira valores deste exemplo fictício para os resultados reais.

### Aprofundamento e cuidados

Agora cada linha é um item de uma transação da base Online Retail. InvoiceNo identifica a transação; StockCode identifica o produto. Um pedido pode ocupar várias linhas. Quantity é a quantidade; UnitPrice é o preço unitário em libras esterlinas. CustomerID é um identificador, não uma medida para tirar média. Country é o país informado para o cliente, não prova de local de entrega.

O curso separa uma visão comercial de vendas positivas: quantidade > 0, preço > 0, data válida e código de fatura que não começa com C. O valor calculado é receita bruta dos itens elegíveis, sem reconciliar devoluções. Não é lucro, receita líquida contábil ou caixa recebido. Cancelamentos e quantidades negativas continuam na base bruta e no diagnóstico.

Linhas sem CustomerID podem entrar na visão de vendas, mas não na segmentação por cliente. Não remova duplicatas exatas automaticamente: sem um identificador único de linha não sabemos se representam repetição indevida. Inspecione a base de verdade: metadados publicados nem sempre refletem todas as ausências observadas.


## Exemplo guiado

In [ ]:
r = pd.read_csv(DADOS/'online_retail.csv', dtype={
    'InvoiceNo':'string','StockCode':'string','CustomerID':'string'})
r['InvoiceDate'] = pd.to_datetime(r['InvoiceDate'], errors='coerce')
cancelada = r['InvoiceNo'].str.upper().str.startswith('C', na=False)
elegivel = (r['Quantity'].gt(0) & r['UnitPrice'].gt(0)
            & r['InvoiceDate'].notna() & ~cancelada)
v = r.loc[elegivel].copy()
v['valor'] = v['Quantity'] * v['UnitPrice']
print('Linhas brutas:',len(r),'Linhas elegíveis:',len(v))
print(r.isna().sum())

## Exercício 07.1
Conte faturas distintas e compare com a quantidade de linhas brutas.

In [ ]:
# Escreva sua resposta aqui


## Exercício 07.2
Conte linhas de cancelamento, quantidade negativa, preço não positivo e cliente ausente. As categorias podem se sobrepor.

In [ ]:
# Escreva sua resposta aqui


## Exercício 07.3
Calcule o valor total da visão positiva e declare moeda e definição.

In [ ]:
# Escreva sua resposta aqui


## Exercício 07.4
Liste os dez países de maior receita nessa visão.

In [ ]:
# Escreva sua resposta aqui


## Exercício 07.5
Calcule ticket médio por fatura; não use a média das linhas de itens.

In [ ]:
# Escreva sua resposta aqui
